## Imports

In [539]:
import json
import re
import requests

import pandas as pd
import numpy as np
from dadata import Dadata

from bs4 import BeautifulSoup

## Functions

In [591]:
def fill_address(value):
    if 'дрес' not in value.lower():
        return np.nan
    else:
        try:
            address = value.split('дрес')[1].split(':')[1].strip()
            if ('назначение' or 'площадь') in address:
                address = ', '.join(map(lambda x: x.strip(), value.split('дрес')[1].split(':')[1].split(',', 5)[:-1]))
            elif not address:
                address = ', '.join(map(lambda x: x.strip(), value.split('дрес')[1].split('у', 1)[1].split(',', 5)[:-1]))
            elif 'общей площадью' in address:
                address =  ', '.join(map(lambda x: x.strip(), value.split('дрес')[1].split('у', 1)[1].split(',', 5)[:-2]))
            
            if len(address.split(',')) > 1:
                return address if address[0] != ':' else address.split(':')[1].strip()
            else:
                return np.nan
        except IndexError:
            return np.nan

In [592]:
def fillna_cadastr_num(x, y):
    pattern = r'\b\d+:\d+:\d+:\d+\b'
    if pd.isnull(x):
        matches = re.findall(pattern, y)
        if matches:
            return matches[0].strip()
        else:
            return np.nan
    else:
        return x

In [593]:
def fillna_address(address, cadastr_num):
    dadata_api = "33a1e9d8e1e0a564c028d7d82e1a908f361b82a8"
    secret_key = "b9b0616923b59bf5179c7108c96bf4b267a9ade1"
    dadata = Dadata(dadata_api, secret_key)

    if pd.isnull(address):
        if not pd.isnull(cadastr_num):
            try:
                result = dadata.find_by_id('address', cadastr_num)[0]['value']
                return result
            except IndexError:
                return np.nan
        else:
            return np.nan
    else:
        return address

In [563]:
# Нашёл ip, по которому можно получить адрес по кадастровому номеру бесплатно
params = {
    'Cookie': r"device_uuid=ab876b06-da70-4fdc-9b0c-0dacb725c80c; _ym_uid=1695065687733068195; _ym_d=1695065687; _gid=GA1.2.815736755.1695065687; tmr_lvid=aca5eb5b75aaee2475d9fa1c571f7d0a; tmr_lvidTS=1695065687460; _ym_isad=1; PAPVisitorId=22e96c8aa7f5af17aaa16232d1RPMDPH; _ym_visorc=w; popmechanic_sbjs_migrations=popmechanic_1418474375998%3D1%7C%7C%7C1471519752600%3D1%7C%7C%7C1471519752605%3D1; aprt_last_partner=actionpay; aprt_last_apclick=; aprt_last_apsource=17322767; tmr_detect=1%7C1695066370206; laravel_session=eyJpdiI6IjdaNjdHUFRueVJKZ3V3OTNvZE95K0E9PSIsInZhbHVlIjoiS0h4S0R3aS9mZ0dFaDh6OVRMVEhuOFpDbDRqRGdvYzRGazVIcWZWYmpEWXRkZDBYM0pPbnNxZlN5YmwzL0tGL3o3T1ppUlVvQmMzd2xyYWJpT2Q0MTdWamgxd1BodHI1MUJacnp1c3RxNk9KQTFOZFlTQ0ZyY0pvSEV1anowU0EiLCJtYWMiOiI1MGZkNmYwMTYxZjRkOWZhNTM4YjliZTUwNjNmODg0MWU2ZGUwYzMyODQzMjY1YjdlMzVlMzNiYzI1NWVhMDEwIiwidGFnIjoiIn0%3D; _gat_UA-127986179-1=1; _gat=1; _ga_6QZMB4SDLP=GS1.1.1695065687.1.1.1695066460.60.0.0; _ga=GA1.1.2031855303.1695065687",
    'User-Agent': r"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
    'Accept': r"application/json, text/plain, */*"
}
example = requests.post(r"https://egrnrstr.ru/search/30:09:070302:687", params=params).json()

In [505]:
data_parts = []

with open(r'json_file_full.json', encoding='utf-8') as f:
    data = pd.read_json(f)

for i in data['content']:
    data_parts.extend(i)

df = pd.DataFrame.from_records(data_parts)

columns = [
    'id',
    'lotStatus',
    'biddType',
    'biddForm',
    'lotName',
    'lotDescription',
    'priceMin',
    'priceFin',
    'biddEndTime',
    'lotImages',
    'category',
    'createDate',
    'characteristics'
]
df = df[columns].reset_index(drop=True)

df['biddEndTime'] = pd.to_datetime(df['biddEndTime']).dt.strftime('%d.%m.%Y %H:%M')
df['createDate'] = pd.to_datetime(df['createDate']).dt.strftime('%d.%m.%Y %H:%M')
df['biddType'] = df['biddType'].apply(lambda x: x['name'])
df['biddForm'] = df['biddForm'].apply(lambda x: x['name'])
df['category'] = df['category'].apply(lambda x: x['name'])
df['area'] = df['characteristics'].apply(
    lambda x: [i['characteristicValue'] for i in x if i['code'] == 'totalAreaRealty'][0] \
                if len([i['characteristicValue'] for i in x if i['code'] == 'totalAreaRealty']) != 0 \
                else np.nan
)
df['cadastral_number'] = df['characteristics'].apply(
    lambda x: [i['characteristicValue'] for i in x if i['code'] == 'cadastralNumberRealty' \
                if 'characteristicValue' in i.keys()][0] if len([i['characteristicValue'] for i in x if i['code'] == 'cadastralNumberRealty' if 'characteristicValue' in i.keys()]) != 0 \
                else np.nan
)

In [510]:
df['address'] = df['lotDescription'].map(fill_address)

In [511]:
df['address'].isna().sum() / len(df)

0.45

In [514]:
df['cadastral_number'] = df.apply(lambda x: fillna_cadastr_num(x['cadastral_number'], x['lotDescription']), axis=1)
df['address'] = df.apply(lambda x: fillna_address(x['address'], x['cadastral_number']), axis=1)

In [515]:
df[df['address'].isna()]['cadastral_number']

29        30:12:010032:345
30       30:12:010032:345 
33       30:12:020348:115 
34     Согласно Извещению 
35      Согласно Извещению
              ...         
673       30:04:040103:317
674       30:09:070302:687
675       30:09:070302:687
676      30:06:020101:1633
677       30:09:070302:687
Name: cadastral_number, Length: 148, dtype: object

In [516]:
df['address'].isna().sum() / len(df)

0.21764705882352942

In [521]:
df

,id,lotStatus,biddType,biddForm,lotName,lotDescription,priceMin,priceFin,biddEndTime,lotImages,category,createDate,characteristics,area,cadastral_number,address
0,21000002210000000426_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Нежилое помещение, общей площадью 23,3 кв. м, ...","Нежилое помещение, расположенное по адресу: Ас...",132000.0,NaN,16.05.2022 20:00,"[625846877a5b5e2bbb394249, 6258468663a456561b6...",Гаражи и машиноместа,14.04.2022 16:00,"[{'characteristicValue': 23.3, 'name': 'Общая ...",23.3,30:12:040286:3355,"Астраханская область, г. Астрахань, р-н Советс..."
1,21000002210000000422_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,Нежилое помещение (мастерск.) общей площадью 1...,"Нежилое помещение (мастерск.), расположенное п...",1284000.0,NaN,16.05.2022 20:00,"[6258446f63a456561b66bfd1, 6258446f7a5b5e2bbb3...",Нежилые помещения,13.04.2022 16:53,"[{'characteristicValue': 148.2, 'name': 'Общая...",148.2,30:12:000000:8455,"Астраханская область, г. Астрахань, р-н Трусов..."
2,21000002210000000627_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Нежилое помещение, общей площадью 23,3 кв. м, ...","Нежилое помещение, расположенное по адресу: Ас...",132000.0,NaN,18.07.2022 20:00,"[62b048621c53e20ba76b9258, 62b048601c53e20ba76...",Нежилые помещения,20.06.2022 10:17,"[{'characteristicValue': 23.3, 'name': 'Общая ...",23.3,30:12:040286:3355,"Астраханская область, г. Астрахань, р-н Советс..."
3,21000002210000000628_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,Нежилое помещение (мастерск.) общей площадью 1...,"Нежилое помещение (мастерск.), расположенное п...",1284000.0,NaN,18.07.2022 20:00,"[62b00e9bc7d464240973602d, 62b00e9bc7d46424097...",Нежилые помещения,20.06.2022 06:13,"[{'characteristicValue': 148.2, 'name': 'Общая...",148.2,30:12:000000:8455,"Астраханская область, г. Астрахань, р-н Трусов..."
4,21000002210000001592_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Нежилое помещение, общей площадью 23,3 кв. м, ...","Нежилое помещение, расположенное по адресу: Ас...",114000.0,NaN,12.12.2022 20:00,"[6374c842981be20cca54cbd3, 6374c841420c495c7b5...",Нежилые помещения,16.11.2022 11:32,"[{'characteristicValue': 23.3, 'name': 'Общая ...",23.3,30:12:040286:3355,"Астраханская область, г. Астрахань, р-н Советс..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
675,22000204810000000001_1,FAILED,"Аренда, безвозмездное пользование, доверительн...",Аукцион,Аренда помещения,"Часть здания (чз2), площадью 15,4 кв.м., в том...",41888.0,NaN,18.08.2023 08:00,[64c237d6fe13694d7d8d030f],Нежилые помещения,27.07.2023 09:40,"[{'characteristicValue': '1986 год', 'name': '...",946.5,30:09:070302:687,NaN
676,23000028690000000001_1,SUCCEED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Здание, площадью 42,2 кв.м., кадастровый №30:0...","Здание, площадью 42,2 кв.м., кадастровый №30:0...",261530.0,261530.0,24.07.2023 13:00,[649ab5659dfe996e4c00a383],Здания,27.06.2023 10:27,"[{'characteristicValue': '-', 'name': 'Год вво...",42.2,30:06:020101:1633,NaN
677,22000204810000000001_3,FAILED,"Аренда, безвозмездное пользование, доверительн...",Аукцион,Аренда помещения,"Часть здания (чз6), площадью 18,1 кв.м., в том...",55929.0,NaN,18.08.2023 08:00,[64c23b80038d2d0093293766],Нежилые помещения,27.07.2023 09:40,"[{'characteristicValue': '1986 год', 'name': '...",946.5,30:09:070302:687,NaN
678,21000033590000000001_1,SUCCEED,Продажа (приватизация) государственного и муни...,Электронный аукцион,Нежилое здание с земельным участком,"Нежилое здание площадью 135,5 кв. м, кадастров...",458000.0,458000.0,29.03.2022 19:00,[621f1eb856e04904d26cee46],Здания,02.03.2022 07:26,"[{'characteristicValue': 135.5, 'name': 'Общая...",135.5,30:10:020401:3033,"Астраханская область, Харабалинский р-н, с. Са..."


In [286]:
df[df['cadastral_number'].isna()]['address'][40]

'г. Астрахань, ул. Пушкина'

In [501]:
df

,id,lotStatus,biddType,biddForm,lotName,lotDescription,priceMin,priceFin,biddEndTime,lotImages,category,createDate,characteristics,area,cadastral_number,address
0,21000002210000000426_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Нежилое помещение, общей площадью 23,3 кв. м, ...","Нежилое помещение, расположенное по адресу: Ас...",132000.0,NaN,16.05.2022 20:00,"[625846877a5b5e2bbb394249, 6258468663a456561b6...",Гаражи и машиноместа,14.04.2022 16:00,"[{'characteristicValue': 23.3, 'name': 'Общая ...",23.3,30:12:040286:3355,None
1,21000002210000000422_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,Нежилое помещение (мастерск.) общей площадью 1...,"Нежилое помещение (мастерск.), расположенное п...",1284000.0,NaN,16.05.2022 20:00,"[6258446f63a456561b66bfd1, 6258446f7a5b5e2bbb3...",Нежилые помещения,13.04.2022 16:53,"[{'characteristicValue': 148.2, 'name': 'Общая...",148.2,30:12:000000:8455,None
2,21000002210000000627_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Нежилое помещение, общей площадью 23,3 кв. м, ...","Нежилое помещение, расположенное по адресу: Ас...",132000.0,NaN,18.07.2022 20:00,"[62b048621c53e20ba76b9258, 62b048601c53e20ba76...",Нежилые помещения,20.06.2022 10:17,"[{'characteristicValue': 23.3, 'name': 'Общая ...",23.3,30:12:040286:3355,None
3,21000002210000000628_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,Нежилое помещение (мастерск.) общей площадью 1...,"Нежилое помещение (мастерск.), расположенное п...",1284000.0,NaN,18.07.2022 20:00,"[62b00e9bc7d464240973602d, 62b00e9bc7d46424097...",Нежилые помещения,20.06.2022 06:13,"[{'characteristicValue': 148.2, 'name': 'Общая...",148.2,30:12:000000:8455,None
4,21000002210000001592_1,FAILED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Нежилое помещение, общей площадью 23,3 кв. м, ...","Нежилое помещение, расположенное по адресу: Ас...",114000.0,NaN,12.12.2022 20:00,"[6374c842981be20cca54cbd3, 6374c841420c495c7b5...",Нежилые помещения,16.11.2022 11:32,"[{'characteristicValue': 23.3, 'name': 'Общая ...",23.3,30:12:040286:3355,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
675,22000204810000000001_1,FAILED,"Аренда, безвозмездное пользование, доверительн...",Аукцион,Аренда помещения,"Часть здания (чз2), площадью 15,4 кв.м., в том...",41888.0,NaN,18.08.2023 08:00,[64c237d6fe13694d7d8d030f],Нежилые помещения,27.07.2023 09:40,"[{'characteristicValue': '1986 год', 'name': '...",946.5,30:09:070302:687,NaN
676,23000028690000000001_1,SUCCEED,Продажа (приватизация) государственного и муни...,Электронный аукцион,"Здание, площадью 42,2 кв.м., кадастровый №30:0...","Здание, площадью 42,2 кв.м., кадастровый №30:0...",261530.0,261530.0,24.07.2023 13:00,[649ab5659dfe996e4c00a383],Здания,27.06.2023 10:27,"[{'characteristicValue': '-', 'name': 'Год вво...",42.2,30:06:020101:1633,NaN
677,22000204810000000001_3,FAILED,"Аренда, безвозмездное пользование, доверительн...",Аукцион,Аренда помещения,"Часть здания (чз6), площадью 18,1 кв.м., в том...",55929.0,NaN,18.08.2023 08:00,[64c23b80038d2d0093293766],Нежилые помещения,27.07.2023 09:40,"[{'characteristicValue': '1986 год', 'name': '...",946.5,30:09:070302:687,NaN
678,21000033590000000001_1,SUCCEED,Продажа (приватизация) государственного и муни...,Электронный аукцион,Нежилое здание с земельным участком,"Нежилое здание площадью 135,5 кв. м, кадастров...",458000.0,458000.0,29.03.2022 19:00,[621f1eb856e04904d26cee46],Здания,02.03.2022 07:26,"[{'characteristicValue': 135.5, 'name': 'Общая...",135.5,30:10:020401:3033,NaN


In [66]:
df[df['address'].notna()]['address'][92]

'Астраханская область, г. Астрахань'

In [74]:
df.iloc[92]['characteristics']

[{'characteristicValue': 332.2,
  'name': 'Общая площадь',
  'code': 'totalAreaRealty',
  'unit': {'code': '081',
   'name': 'Квадратный метр общей площади',
   'symbol': 'м^2 общ. пл'},
  'type': 'Decimal'},
 {'name': 'Материалы наружных стен здания ',
  'code': 'buildingWallMaterials',
  'type': 'Text(100)'},
 {'characteristicValue': '4',
  'name': 'Количество этажей ',
  'code': 'numberFloors',
  'type': 'Text(100)'},
 {'name': 'Количество подземных этажей ',
  'code': 'numberUndergroundFloors',
  'type': 'Text(100)'},
 {'characteristicValue': 'нежилое',
  'name': 'Назначение здания ',
  'code': 'purposeBuilding',
  'type': 'Text(100)'},
 {'characteristicValue': 'не зарегистрированы',
  'name': 'Общие сведения об ограничениях и обременениях\xa0',
  'code': 'restrictionsEncumbrances',
  'type': 'Text(500)'},
 {'name': 'Вид ограничений и обременений',
  'code': 'typeRestrictionsEncumbrances',
  'type': 'Text(100)'},
 {'name': 'Кадастровая стоимость ', 'code': 'cadastralValue', 'type':

In [180]:
df[df['address'].isna()]['lotDescription'].index[:50]

Index([ 29,  30,  31,  32,  33,  34,  35,  36,  48,  49,  50,  51,  65,  66,
        67,  77, 134, 158, 160, 161, 162, 163, 172, 177, 193, 215, 220, 221,
       224, 227, 236, 237, 238, 281, 283, 284, 288, 289, 290, 291, 292, 293,
       294, 295, 296, 297, 298, 299, 300, 301],
      dtype='int64')

In [ ]:
', '.join(map(lambda x: x.strip(), value.split('дрес')[1].split(':')[1].split(',', 5)[:-1]))

KeyError: 40

In [287]:
df[df['cadastral_number'].isna()]['address'][40].split('дрес')[1].split('у', 1)[1].split(',', 5)[:-2]

IndexError: list index out of range

In [223]:
', '.join(map(lambda x: x.strip(), df[df['address'].isna()]['lotDescription'][134].split('дрес')[1].split(':')[1].split(',', 5)[:-1]))

True

In [338]:
df['lotDescription'][40].split('дрес')[1].strip()

'у: г. Астрахань, ул. Пушкина, 54'

In [334]:
df['lotDescription'][50]

'Год завершения строительства 1970.  Фундамент кирпичный, ленточный; стены и перегородки кирпичные; пе-рекрытия деревянные, утепленные; кровля - асбоцементные листы; полы доща-тые, линолеумовые.'

In [323]:
('назначение' or 'площадь') in 'Адрес: блаблабла, 5аю назначение какое-то'

True

In [48]:
df[(df['cadastral_number'].isna()) & (df['id'] == '21000014170000000069_23')]['lotDescription'].values[0]

'Лот №23-2023/437-2АГ от 20.07.23, уведомление Сыз №2-000023 от 07.07.23, собственник Егоров В.Б. Квартира, площадь: 43,2 кв.м., жилое, кадастровый № 63:08:0107017:3127, этаж № 2, адрес: Самарская обл., г.о. Сызрань, г. Сызрань, ул. Астраханская, д. 25, кв. 46. Зарегистрировано 1 совершеннолетнее лицо.'

In [481]:
df_nan = df[df['address'].isna()].reset_index(drop=True)

In [455]:
df['address_new'] = df.apply(lambda x: x['address'], axis=1)

In [474]:
df['address'] = df.apply(lambda x: fillna_address(address=x['address'], cadastr_num=x['cadastral_number']), axis=1)

In [314]:
df['address'] = df['lotDescription'].map(fill_address)

In [308]:
df['address'] = df['address'].map(fill_na_address)